# 🧬 PhagePipeline — Official Google Colab Deployment Notebook

**Therapeutic Phage Candidate Selection Pipeline**  
*Astraphage Innovations*

---

## About This Notebook

This is the **official, production-grade Google Colab version** of the PhagePipeline Snakemake workflow.  
It is fully portable — no WSL paths, no local user paths, no hardcoded credentials, no assumptions about any specific Google Drive structure beyond your chosen root folder.

### What This Notebook Does

| Step | Action |
|------|--------|
| 1 | **Mounts Google Drive** — persistent storage for databases, inputs, and outputs |
| 2 | **Installs Miniforge** (Conda + Mamba) in the Colab runtime |
| 3 | **Installs Snakemake** in the base Conda environment |
| 4 | **Clones the repository** — supports private (PAT), public, or ZIP upload |
| 5 | **Creates symlinks** — links `databases/`, `raw_input/`, `results/`, `reports/` to Google Drive |
| 6 | **Creates all Conda environments** from repository YAML files |
| 7 | **Validates everything** — databases, FASTAs, envs, symlinks, disk, RAM, GPU, tools |
| 8 | **Executes the Snakemake workflow** exactly as implemented — no rule changes |
| 9 | **Displays professional reports** — scoring tables, genome maps, protein tables, candidate reports |

### Pipeline Stages

| Stage | Tools | Conda env |
|-------|-------|-----------|
| **Stage 0 — Quality** | CheckV | `env_checkv` |
| **Stage 1 — Annotation** | Pharokka → PHOLD → Phynteny | `env_pharokka`, `pholdENV`, `phyntenyENV` |
| **Stage 1 — Extras** | PhageDPO · PhageRBPdetect v4 · PhageTailFinder | `env_phage_ml` |
| **Stage 2 — Lifestyle** | BACPHLIP | `env_bacphlip` |
| **Stage 2 — Safety** | RGI · Abricate-CARD · Abricate-VFDB · AcrDB-BLASTP | `env_rgi`, `env_abricate`, `env_acrdb` |
| **Stage 3 — Post-processing** | build_annotated_gb · build_protein_table · visualize_genome · summarize_sample | `phage` |
| **Stage 4 — Scoring** | score_and_rank | `phage` |

### Databases on Google Drive

| Folder name (under `databases/`) | Wrapper script variable | Tool |
|-----------------------------------|------------------------|------|
| `checkv-db/` | `DB` in `run_checkv.sh` | CheckV |
| `pharokka_db/` | `DB` in `run_pharokka.sh` | Pharokka |
| `phold_db/` | `DB_PATH` in `run_phold.sh` | PHOLD |
| `phynteny_models/models/` | `MODELS_PATH` in `run_phynteny.sh` | Phynteny |
| `acrdb_db/122_KnownAcr/Known_Acr.faa` | `DB` in `run_acrdb_blast.sh` · `acrdb_db` in `config.yaml` | AcrDB |

> **Note:** Abricate (CARD, VFDB) and RGI use bundled databases that are downloaded when the tools are installed. No external database path is needed for those.

---

## ⚡ Quick Start

1. **Edit Cell 2** (the only cell you need to change)
2. **Runtime → Run all** — or run cells one by one in order
3. Results appear in Section 8


In [ ]:
# ==============================================================================
# CELL 2 — USER CONFIGURATION
# THIS IS THE ONLY CELL YOU NEED TO EDIT.
# Everything else is detected automatically from the repository.
# ==============================================================================

# ── Repository clone method ────────────────────────────────────────────────────
# "token"  — private GitHub repository, requires GITHUB_PAT below
# "public" — public GitHub repository, no token needed
# "zip"    — skip cloning; upload a ZIP of the repository to Colab first
CLONE_METHOD = "token"   # <── EDIT: "token" | "public" | "zip"

# GitHub repository URL (HTTPS, no credentials embedded)
REPO_URL    = "https://github.com/CHANGEME/PhagePipeline"  # <── EDIT
REPO_BRANCH = "main"                                        # <── EDIT if needed

# GitHub Personal Access Token (used only when CLONE_METHOD = "token")
# Generate at: https://github.com/settings/tokens  → classic → repo scope
GITHUB_PAT  = ""   # <── PASTE TOKEN HERE  (do NOT commit this)

# Path to uploaded ZIP (used only when CLONE_METHOD = "zip")
# Upload via Colab Files panel (left sidebar) before running this cell.
REPO_ZIP_PATH = "/content/PhagePipeline.zip"   # <── EDIT if needed

# ── Google Drive root ──────────────────────────────────────────────────────────
# Name of the folder in the ROOT of your Google Drive that stores pipeline data.
# Subfolders created automatically: databases/, raw_input/, results/, reports/
DRIVE_ROOT = "PhagePipeline"   # <── EDIT to your preferred folder name

# ── Samples ───────────────────────────────────────────────────────────────────
# Comma-separated sample names (no .fasta extension).
# Each sample must have a corresponding {sample}.fasta in raw_input/.
SAMPLES = "PhageA,PhageB"   # <── EDIT

# ── Run ID ────────────────────────────────────────────────────────────────────
# Namespace for reports/{run_id}/. Leave empty to auto-use today's date.
RUN_ID = ""   # <── EDIT if needed  e.g. "2026-07-15_batch1"

# ── Compute ───────────────────────────────────────────────────────────────────
THREADS = 4   # <── Number of CPU threads (Colab free tier: 2-4)

# ── Advanced (leave defaults unless you know what you're doing) ───────────────
# Extra Snakemake flags, e.g. "--dry-run", "--forceall", "--rerun-incomplete"
SNAKEMAKE_EXTRA_FLAGS = ""
# Force re-create Conda environments (slow; use only when debugging env issues)
FORCE_RECREATE_ENVS = False

# ==============================================================================
print("✅ Configuration loaded.")
print(f"   Clone method : {CLONE_METHOD}")
print(f"   Repo URL     : {REPO_URL}")
print(f"   Branch       : {REPO_BRANCH}")
print(f"   Drive root   : /content/drive/MyDrive/{DRIVE_ROOT}/")
print(f"   Samples      : {SAMPLES}")
print(f"   Threads      : {THREADS}")
if GITHUB_PAT:
    print(f"   PAT          : {'*' * len(GITHUB_PAT)} (set)")
else:
    print(f"   PAT          : (not set)")


---
## Section 1 — Runtime Bootstrap

Mounts Google Drive, installs Conda/Mamba, and installs Snakemake.

> **Colab tip:** Enable GPU runtime for faster PHOLD / PhageRBPdetect v4 inference:  
> `Runtime → Change runtime type → T4 GPU`


In [ ]:
# ==============================================================================
# CELL 3 — Mount Google Drive
# ==============================================================================
from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount("/content/drive", force_remount=False)

DRIVE_MOUNT     = "/content/drive/MyDrive"
DRIVE_ROOT_PATH = os.path.join(DRIVE_MOUNT, DRIVE_ROOT)

# Create persistent directory skeleton if it does not exist
for subdir in ["databases", "raw_input", "results", "reports"]:
    os.makedirs(os.path.join(DRIVE_ROOT_PATH, subdir), exist_ok=True)

print(f"\n✅ Google Drive mounted")
print(f"   Pipeline Drive root: {DRIVE_ROOT_PATH}")
print("\nDrive directory contents:")
for entry in sorted(os.listdir(DRIVE_ROOT_PATH)):
    full = os.path.join(DRIVE_ROOT_PATH, entry)
    tag  = "DIR" if os.path.isdir(full) else "FILE"
    print(f"   [{tag}] {entry}")


In [ ]:
# ==============================================================================
# CELL 4 — Install Miniforge (Conda + Mamba)
# ==============================================================================
import subprocess, os, sys

CONDA_PATH = "/opt/conda"
CONDA_BIN  = f"{CONDA_PATH}/bin/conda"
MAMBA_BIN  = f"{CONDA_PATH}/bin/mamba"

def _run(cmd, check=True):
    """Run a shell command, print output, raise on failure."""
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout.strip():
        print(r.stdout.rstrip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed (exit {r.returncode}):\n{cmd}")
    return r

if os.path.exists(CONDA_BIN):
    print(f"✅ Conda already installed at {CONDA_PATH}")
else:
    print("Installing Miniforge3...")
    _run("""
        wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh \
            -O /tmp/miniforge.sh && \
        bash /tmp/miniforge.sh -b -p /opt/conda && \
        rm /tmp/miniforge.sh
    """)

# Ensure conda/mamba are on PATH for all subsequent subprocess calls
os.environ["PATH"] = f"{CONDA_PATH}/bin:" + os.environ.get("PATH", "")
os.environ["CONDA_EXE"] = CONDA_BIN

_run(f"{CONDA_BIN} --version")
_run(f"{MAMBA_BIN} --version")
print("\n✅ Conda + Mamba ready.")


In [ ]:
# ==============================================================================
# CELL 5 — Install Snakemake
# ==============================================================================
import subprocess, os

CONDA_PATH = "/opt/conda"
MAMBA_BIN  = f"{CONDA_PATH}/bin/mamba"

def _run(cmd, check=True):
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout.strip():
        print(r.stdout.rstrip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed (exit {r.returncode}):\n{cmd}")
    return r

r = subprocess.run("snakemake --version", shell=True, capture_output=True, text=True)
if r.returncode == 0:
    print(f"✅ Snakemake already available: {r.stdout.strip()}")
else:
    print("Installing Snakemake into base environment via mamba...")
    _run(f"""
        {MAMBA_BIN} install -n base -y \
            -c conda-forge -c bioconda \
            'snakemake>=7.0' 'snakemake-minimal>=7.0'
    """)
    r2 = subprocess.run("snakemake --version", shell=True, capture_output=True, text=True)
    if r2.returncode == 0:
        print(f"✅ Snakemake installed: {r2.stdout.strip()}")
    else:
        raise RuntimeError("Snakemake installation failed.")

print("\n✅ Snakemake ready.")


---
## Section 2 — Repository Setup

Clones or extracts the PhagePipeline repository into `/content/PhagePipeline`.

### Clone methods

| Method | When to use | What to set |
|--------|------------|-------------|
| `token` | Private GitHub repo | Set `GITHUB_PAT` in Cell 2 |
| `public` | Public GitHub repo | Just set `REPO_URL` and `REPO_BRANCH` |
| `zip` | Can't use GitHub from Colab | Upload ZIP to Colab Files panel first |


In [ ]:
# ==============================================================================
# CELL 6 — Clone / Extract Repository
# ==============================================================================
import os, subprocess, shutil

REPO_DIR = "/content/PhagePipeline"

def _run(cmd, cwd=None, check=True):
    r = subprocess.run(cmd, shell=True, text=True, cwd=cwd,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout.strip():
        print(r.stdout.rstrip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed:\n  {cmd}\n{r.stdout}")
    return r

if CLONE_METHOD == "token":
    # ── Method 1: private repo via GitHub Personal Access Token ─────────────
    if not GITHUB_PAT:
        raise ValueError(
            "CLONE_METHOD='token' requires GITHUB_PAT to be set.\n"
            "Generate one at: https://github.com/settings/tokens  (classic, repo scope)\n"
            "Paste it into GITHUB_PAT in the Configuration cell."
        )
    if "github.com/" not in REPO_URL:
        raise ValueError(f"REPO_URL does not look like a GitHub HTTPS URL: {REPO_URL}")
    repo_path = REPO_URL.replace("https://", "").replace("http://", "")
    auth_url  = f"https://x-access-token:{GITHUB_PAT}@{repo_path}"

    if os.path.exists(REPO_DIR):
        print(f"Repository exists at {REPO_DIR} — pulling latest changes...")
        _run(f"git remote set-url origin '{auth_url}'", cwd=REPO_DIR, check=False)
        _run(f"git pull origin {REPO_BRANCH}", cwd=REPO_DIR)
    else:
        print("Cloning private repository (token auth)...")
        _run(f"git clone --branch '{REPO_BRANCH}' --depth 1 '{auth_url}' '{REPO_DIR}'")

elif CLONE_METHOD == "public":
    # ── Method 2: public repo, no authentication ────────────────────────────
    if os.path.exists(REPO_DIR):
        print(f"Repository exists at {REPO_DIR} — pulling latest changes...")
        _run(f"git pull origin {REPO_BRANCH}", cwd=REPO_DIR)
    else:
        print("Cloning public repository...")
        _run(f"git clone --branch '{REPO_BRANCH}' --depth 1 '{REPO_URL}' '{REPO_DIR}'")

elif CLONE_METHOD == "zip":
    # ── Method 3: ZIP file uploaded to Colab ────────────────────────────────
    if not os.path.exists(REPO_ZIP_PATH):
        raise FileNotFoundError(
            f"ZIP not found at: {REPO_ZIP_PATH}\n"
            "Upload your repository ZIP via the Colab Files panel "
            "(left sidebar → folder icon → upload button)."
        )
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    _run(f"unzip -q '{REPO_ZIP_PATH}' -d /tmp/_repo_extract")
    extracted = os.listdir("/tmp/_repo_extract")
    if len(extracted) == 1 and os.path.isdir(f"/tmp/_repo_extract/{extracted[0]}"):
        shutil.move(f"/tmp/_repo_extract/{extracted[0]}", REPO_DIR)
    else:
        shutil.move("/tmp/_repo_extract", REPO_DIR)
    shutil.rmtree("/tmp/_repo_extract", ignore_errors=True)

else:
    raise ValueError(f"Unknown CLONE_METHOD: '{CLONE_METHOD}'. Must be 'token', 'public', or 'zip'.")

# ── Verify repository integrity ──────────────────────────────────────────────
required = ["Snakefile", "config.yaml", "scripts", "configs", "env", "tools"]
missing  = [f for f in required if not os.path.exists(os.path.join(REPO_DIR, f))]
if missing:
    raise RuntimeError(
        f"Repository appears incomplete. Missing items: {missing}\n"
        f"Verify that {REPO_DIR} contains the full PhagePipeline repository."
    )

print(f"\n✅ Repository ready at: {REPO_DIR}")
print("\nTop-level contents:")
for entry in sorted(os.listdir(REPO_DIR)):
    full = os.path.join(REPO_DIR, entry)
    tag  = "DIR " if os.path.isdir(full) else "FILE"
    print(f"   [{tag}] {entry}")


---
## Section 3 — Symlink Setup

Links persistent Google Drive folders into the repository directory. The pipeline source code is never modified.

```
/content/PhagePipeline/
  databases/ ──→  /content/drive/MyDrive/{DRIVE_ROOT}/databases/   (databases persist)
  raw_input/ ──→  /content/drive/MyDrive/{DRIVE_ROOT}/raw_input/   (FASTAs persist)
  results/   ──→  /content/drive/MyDrive/{DRIVE_ROOT}/results/     (tool outputs persist)
  reports/   ──→  /content/drive/MyDrive/{DRIVE_ROOT}/reports/     (reports persist)
```


In [ ]:
# ==============================================================================
# CELL 7 — Create Symlinks: Repository ↔ Google Drive
# ==============================================================================
import os, shutil

REPO_DIR        = "/content/PhagePipeline"
DRIVE_ROOT_PATH = f"/content/drive/MyDrive/{DRIVE_ROOT}"

# Mapping: (repo_subdir, drive_subdir)
# Only these four dirs persist; everything else in the repo is re-cloned.
SYMLINK_MAP = [
    ("databases", "databases"),
    ("raw_input", "raw_input"),
    ("results",   "results"),
    ("reports",   "reports"),
]

print("Setting up symlinks: repository ↔ Google Drive")
print(f"   Drive root : {DRIVE_ROOT_PATH}")
print(f"   Repo dir   : {REPO_DIR}")
print()

for repo_sub, drive_sub in SYMLINK_MAP:
    repo_path  = os.path.join(REPO_DIR, repo_sub)
    drive_path = os.path.join(DRIVE_ROOT_PATH, drive_sub)

    # Ensure the Drive-side target exists
    os.makedirs(drive_path, exist_ok=True)

    # Remove stale symlink or repo-side directory
    if os.path.islink(repo_path):
        os.unlink(repo_path)
    elif os.path.isdir(repo_path):
        # Preserve any files that were in the repo clone (e.g. .gitkeep)
        for item in os.listdir(repo_path):
            src = os.path.join(repo_path, item)
            dst = os.path.join(drive_path, item)
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(repo_path)

    os.symlink(drive_path, repo_path)

    # Verify
    ok = os.path.islink(repo_path) and os.path.isdir(repo_path)
    if ok:
        target = os.readlink(repo_path)
        print(f"   ✅ {repo_sub:<12}  →  {target}")
    else:
        raise RuntimeError(f"Symlink creation failed: {repo_path} → {drive_path}")

print("\n✅ All symlinks verified.")
print("   Pipeline reads/writes through repo paths; data persists on Google Drive.")


---
## Section 4 — Conda Environment Setup

Creates all Conda environments required by the pipeline from the repository YAML files.

**YAML-based (from repository):**

| Environment | YAML file | Wrapper script |
|-------------|-----------|---------------|
| `env_checkv` | `configs/checkv.yaml` | `scripts/checkv/run_checkv.sh` |
| `env_pharokka` | `configs/pharokka.yml` | `scripts/pharokka/run_pharokka.sh` |
| `pholdENV` | `env/phold_env.yml` | `scripts/phage/run_phold.sh` |
| `phyntenyENV` | `env/phynteny_env.yml` | `scripts/phage/run_phynteny.sh` |
| `env_phage_ml` | `configs/env_phage_ml.yml` | PhageDPO · PhageRBPdetect v4 · PhageTailFinder |

**Manually specified (no YAML in repository — see `docs/Installation.md`):**

| Environment | Wrapper script |
|-------------|---------------|
| `phage` | `build_annotated_gb.py`, `build_protein_table.py`, `visualize_genome.py`, `summarize_sample.py`, `score_and_rank.py` |
| `env_bacphlip` | `scripts/bacphlip/run_bacphlip.sh` |
| `env_rgi` | `scripts/rgi/run_rgi.sh` |
| `env_abricate` | `scripts/abricate/run_abricate.sh` |
| `env_acrdb` | `scripts/acrdb/run_acrdb_blast.sh` |


In [ ]:
# ==============================================================================
# CELL 8 — Create Conda Environments
# ==============================================================================
import subprocess, os

REPO_DIR   = "/content/PhagePipeline"
MAMBA_BIN  = "/opt/conda/bin/mamba"
CONDA_BIN  = "/opt/conda/bin/conda"

def _conda_env_exists(name):
    r = subprocess.run(f"{CONDA_BIN} env list", shell=True,
                       capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if line.startswith(name + " ") or line.startswith(name + "\t"):
            return True
    return False

def _create_from_yaml(env_name, yaml_path, force=False):
    if not os.path.exists(yaml_path):
        raise FileNotFoundError(f"Environment YAML not found: {yaml_path}")
    if _conda_env_exists(env_name) and not force:
        print(f"   ⏩  '{env_name}' already exists — skipping")
        return
    print(f"   ⚙️   Creating '{env_name}' from {os.path.basename(yaml_path)} ...")
    cmd = f"{MAMBA_BIN} env create -f '{yaml_path}' -y"
    if force:
        cmd = f"{MAMBA_BIN} env create -f '{yaml_path}' --force -y"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:])
        print(r.stderr[-2000:])
        raise RuntimeError(f"Failed to create environment '{env_name}'")
    print(f"   ✅  '{env_name}' created")

def _create_manual(env_name, channels, packages, force=False):
    if _conda_env_exists(env_name) and not force:
        print(f"   ⏩  '{env_name}' already exists — skipping")
        return
    short_pkgs = ', '.join(packages[:3]) + ('...' if len(packages) > 3 else '')
    print(f"   ⚙️   Creating '{env_name}' ({short_pkgs}) ...")
    ch  = " ".join(f"-c {c}" for c in channels)
    pkg = " ".join(f"'{p}'" for p in packages)
    cmd = f"{MAMBA_BIN} create -n '{env_name}' {ch} {pkg} -y"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:])
        print(r.stderr[-2000:])
        raise RuntimeError(f"Failed to create environment '{env_name}'")
    print(f"   ✅  '{env_name}' created")

force = FORCE_RECREATE_ENVS

# ── YAML-based (from repository) ─────────────────────────────────────────────
print("[YAML-based environments — from repository]")
_create_from_yaml("env_checkv",  os.path.join(REPO_DIR, "configs", "checkv.yaml"),    force)
_create_from_yaml("env_pharokka",os.path.join(REPO_DIR, "configs", "pharokka.yml"),   force)
_create_from_yaml("pholdENV",    os.path.join(REPO_DIR, "env",     "phold_env.yml"),  force)
_create_from_yaml("phyntenyENV", os.path.join(REPO_DIR, "env",     "phynteny_env.yml"), force)
_create_from_yaml("env_phage_ml",os.path.join(REPO_DIR, "configs", "env_phage_ml.yml"), force)

# ── Manually specified (no YAML in repository) ────────────────────────────────
print("\n[Manually-specified environments — per docs/Installation.md]")

# 'phage' — used by all post-processing Python scripts:
#   build_annotated_gb.py  (Bio.SeqIO)
#   build_protein_table.py (Bio.SeqIO)
#   visualize_genome.py    (pygenomeviz, matplotlib)
#   summarize_sample.py    (csv, os — stdlib only)
#   score_and_rank.py      (csv, os — stdlib only)
_create_manual(
    "phage",
    channels=["conda-forge", "bioconda"],
    packages=["python=3.10", "biopython>=1.80", "pygenomeviz>=0.4",
              "matplotlib>=3.6", "pandas>=1.5", "numpy>=1.23"],
    force=force
)

# 'env_bacphlip' — run_bacphlip.sh calls:
#   conda run -n env_bacphlip bacphlip -i ...
#   The wrapper also calls $(which hmmsearch), so hmmer must be installed.
_create_manual(
    "env_bacphlip",
    channels=["conda-forge", "bioconda"],
    packages=["python=3.8", "bacphlip", "hmmer>=3.3"],
    force=force
)

# 'env_rgi' — run_rgi.sh calls:
#   conda run -n env_rgi rgi main --local ...
#   The --local flag means RGI uses its own bundled CARD database.
_create_manual(
    "env_rgi",
    channels=["conda-forge", "bioconda"],
    packages=["python=3.9", "rgi>=6.0", "diamond>=2.0"],
    force=force
)

# 'env_abricate' — run_abricate.sh calls:
#   conda run -n env_abricate abricate --db card <genome>
#   Abricate bundles CARD, VFDB internally — no external DB path needed.
_create_manual(
    "env_abricate",
    channels=["conda-forge", "bioconda"],
    packages=["abricate"],
    force=force
)

# 'env_acrdb' — run_acrdb_blast.sh calls:
#   conda run -n env_acrdb blastp -query ... -db databases/acrdb_db/122_KnownAcr/Known_Acr.faa
#   The database FASTA must be pre-indexed with makeblastdb (see validation cell).
_create_manual(
    "env_acrdb",
    channels=["conda-forge", "bioconda"],
    packages=["blast>=2.14"],
    force=force
)

print("\n" + "="*60)
print("✅ All Conda environments processed.")
print("\nInstalled environments:")
r = subprocess.run(f"{CONDA_BIN} env list", shell=True, capture_output=True, text=True)
PIPELINE_ENVS = {
    "env_checkv", "env_pharokka", "pholdENV", "phyntenyENV",
    "env_phage_ml", "phage", "env_bacphlip", "env_rgi",
    "env_abricate", "env_acrdb"
}
for line in r.stdout.splitlines():
    if line.startswith("#") or not line.strip():
        continue
    env_name = line.split()[0]
    marker = "  ◀ pipeline" if env_name in PIPELINE_ENVS else ""
    print(f"   {line}{marker}")


In [ ]:
# ==============================================================================
# CELL 9 — Make Wrapper Scripts Executable + Index AcrDB
# ==============================================================================
import subprocess, os

REPO_DIR  = "/content/PhagePipeline"
CONDA_BIN = "/opt/conda/bin/conda"

# All wrapper scripts discovered from the repository
WRAPPER_SCRIPTS = [
    "scripts/checkv/run_checkv.sh",
    "scripts/pharokka/run_pharokka.sh",
    "scripts/phage/run_phold.sh",
    "scripts/phage/run_phynteny.sh",
    "scripts/bacphlip/run_bacphlip.sh",
    "scripts/rgi/run_rgi.sh",
    "scripts/abricate/run_abricate.sh",
    "scripts/acrdb/run_acrdb_blast.sh",
    "scripts/phagedpo/run_phagedpo.sh",
    "scripts/phagerbpdetect/run_phagerbpdetect_v4.sh",
    "scripts/phagetailfinder/run_phagetailfinder.sh",
    "scripts/run_all_ml_tools.sh",
]

print("Setting execute permissions on wrapper scripts...")
all_ok = True
for rel in WRAPPER_SCRIPTS:
    full = os.path.join(REPO_DIR, rel)
    if os.path.exists(full):
        os.chmod(full, 0o755)
        print(f"   ✅ {rel}")
    else:
        print(f"   ⚠️  NOT FOUND: {rel}")
        all_ok = False

# ── Index AcrDB FASTA for BLAST ───────────────────────────────────────────────
# run_acrdb_blast.sh uses: blastp -db databases/acrdb_db/122_KnownAcr/Known_Acr.faa
# BLAST requires a pre-built index. Create it if not already indexed.
ACRDB_FASTA = os.path.join(REPO_DIR, "databases", "acrdb_db", "122_KnownAcr", "Known_Acr.faa")
ACRDB_INDEX = ACRDB_FASTA + ".pdb"   # presence of .pdb indicates indexed

if os.path.exists(ACRDB_FASTA):
    if os.path.exists(ACRDB_INDEX):
        print("\n   ✅ AcrDB BLAST index already exists")
    else:
        print("\n   Indexing AcrDB FASTA for BLAST...")
        r = subprocess.run(
            f"{CONDA_BIN} run -n env_acrdb makeblastdb "
            f"-in '{ACRDB_FASTA}' -dbtype prot -parse_seqids",
            shell=True, capture_output=True, text=True
        )
        if r.returncode == 0:
            print("   ✅ AcrDB indexed successfully")
        else:
            print(f"   ⚠️  AcrDB indexing failed:\n{r.stderr}")
else:
    print("\n   ⚠️  AcrDB FASTA not found — skipping indexing")
    print(f"      Expected: {ACRDB_FASTA}")

# ── Update Abricate databases ─────────────────────────────────────────────────
# Abricate bundles CARD and VFDB, but they may need updating after fresh install.
print("\n   Updating Abricate built-in databases (CARD, VFDB)...")
r_ab = subprocess.run(
    f"{CONDA_BIN} run -n env_abricate abricate --setupdb 2>&1",
    shell=True, capture_output=True, text=True
)
if r_ab.returncode == 0:
    print("   ✅ Abricate databases ready")
else:
    print(f"   ⚠️  Abricate setupdb returned non-zero (may still work):\n{r_ab.stdout[:500]}")

# ── Update RGI CARD local database ────────────────────────────────────────────
# run_rgi.sh uses --local flag, meaning RGI manages its own local CARD copy.
print("\n   Loading RGI CARD local database...")
r_rgi = subprocess.run(
    f"{CONDA_BIN} run -n env_rgi rgi load --local 2>&1 | tail -3",
    shell=True, capture_output=True, text=True
)
print(f"   RGI load: {r_rgi.stdout.strip()[:200] if r_rgi.stdout.strip() else '(no output)'}")

if not all_ok:
    print("\n⚠️  Some wrapper scripts were not found. Verify the repository clone is complete.")
else:
    print("\n✅ All wrapper scripts executable.")


---
## Section 5 — Pre-flight Validation

Comprehensive diagnostics before running the pipeline. All checks report **PASS / WARN / FAIL**.

- **FAIL** = pipeline cannot run until fixed
- **WARN** = pipeline may run but some features could fail
- **PASS** = all good


In [ ]:
# ==============================================================================
# CELL 10 — Pre-flight Validation
# ==============================================================================
import os, subprocess, shutil

try:
    import psutil
except ImportError:
    subprocess.run("pip install -q psutil", shell=True)
    import psutil

REPO_DIR        = "/content/PhagePipeline"
DRIVE_ROOT_PATH = f"/content/drive/MyDrive/{DRIVE_ROOT}"
CONDA_BIN       = "/opt/conda/bin/conda"

# ── Validation framework ──────────────────────────────────────────────────────
checks = []   # list of (category, name, status, detail)

def _check(cat, name, passed, warn=False, detail=""):
    if passed:      status, icon = "PASS", "✅"
    elif warn:      status, icon = "WARN", "⚠️ "
    else:           status, icon = "FAIL", "❌"
    checks.append((cat, name, status, detail))
    suf = f" — {detail}" if detail else ""
    print(f"   {icon} [{status}] {name}{suf}")

def _section(title):
    print(f"\n{'─'*62}")
    print(f"  {title}")
    print(f"{'─'*62}")

def _conda_env_exists(name):
    r = subprocess.run(f"{CONDA_BIN} env list", shell=True, capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if line.startswith(name + " ") or line.startswith(name + "\t"):
            return True
    return False

print("=" * 62)
print("  PhagePipeline — Pre-flight Validation")
print("=" * 62)

# ── 1. System Resources ───────────────────────────────────────────────────────
_section("1. System Resources")
cpu_n   = os.cpu_count() or 1
ram_gb  = psutil.virtual_memory().total / (1024**3)
colab_free = psutil.disk_usage("/content").free / (1024**3)
drive_free = psutil.disk_usage("/content/drive").free / (1024**3)

_check("System", f"CPU cores: {cpu_n}",
       cpu_n >= 2, warn=cpu_n < 2,
       detail="≥2 needed for Snakemake parallel rules")
_check("System", f"RAM: {ram_gb:.1f} GB",
       ram_gb >= 8, warn=(4 <= ram_gb < 8),
       detail="≥8 GB recommended for PHOLD + PhageRBPdetect")
_check("System", f"Colab /content free: {colab_free:.1f} GB",
       colab_free >= 20, warn=(10 <= colab_free < 20),
       detail="≥20 GB recommended (Conda envs + temp files)")
_check("System", f"Google Drive free: {drive_free:.1f} GB",
       drive_free >= 10, warn=(5 <= drive_free < 10))

gpu_r = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
    shell=True, capture_output=True, text=True
)
if gpu_r.returncode == 0 and gpu_r.stdout.strip():
    _check("System", f"GPU: {gpu_r.stdout.strip()}", True,
           detail="Accelerates PHOLD and PhageRBPdetect v4")
else:
    _check("System", "GPU: not available (CPU-only mode)", True, warn=True,
           detail="Enable via Runtime → Change runtime type → T4 GPU")

# ── 2. Repository Integrity ───────────────────────────────────────────────────
_section("2. Repository Integrity")
# Core files inferred from Snakefile, config.yaml, and wrapper scripts
CORE_FILES = [
    "Snakefile",
    "config.yaml",
    # Conda env YAMLs
    "configs/checkv.yaml",
    "configs/pharokka.yml",
    "configs/env_phage_ml.yml",
    "env/phold_env.yml",
    "env/phynteny_env.yml",
    # Wrapper scripts
    "scripts/checkv/run_checkv.sh",
    "scripts/pharokka/run_pharokka.sh",
    "scripts/phage/run_phold.sh",
    "scripts/phage/run_phynteny.sh",
    "scripts/bacphlip/run_bacphlip.sh",
    "scripts/rgi/run_rgi.sh",
    "scripts/abricate/run_abricate.sh",
    "scripts/acrdb/run_acrdb_blast.sh",
    "scripts/phagedpo/run_phagedpo.sh",
    "scripts/phagerbpdetect/run_phagerbpdetect_v4.sh",
    "scripts/phagetailfinder/run_phagetailfinder.sh",
    # Python post-processing scripts
    "scripts/build_annotated_gb.py",
    "scripts/build_protein_table.py",
    "scripts/visualize_genome.py",
    "scripts/summarize_sample.py",
    "scripts/score_and_rank.py",
    "scripts/phagetailfinder/format_fasta.py",
]
for rel in CORE_FILES:
    _check("Repo", rel, os.path.exists(os.path.join(REPO_DIR, rel)))

# Vendored tools (tools/ directory — experimental)
TOOL_FILES = [
    ("tools/PhageRBPdetection/PhageRBPdetect_v4/PhageRBPdetect_v4_inference.py",
     "PhageRBPdetect v4 inference script"),
    ("tools/PhageRBPdetection/data/RBPdetect_v4_ESMfine",
     "PhageRBPdetect v4 model"),
    ("tools/PhageTailFinder/scripts/predict.py",
     "PhageTailFinder predict script"),
    ("tools/PhageTailFinder/dbs/tail_pfam",
     "PhageTailFinder tail HMM database"),
    ("tools/PhageTailFinder/dbs/nontail_pfam",
     "PhageTailFinder nontail HMM database"),
    ("tools/phagedpo/phagedpo_cli.py",
     "PhageDPO CLI script"),
]
for rel, desc in TOOL_FILES:
    _check("Tools", rel,
           os.path.exists(os.path.join(REPO_DIR, rel)),
           warn=True, detail=f"Experimental: {desc}")

# ── 3. Symlink Verification ───────────────────────────────────────────────────
_section("3. Symlink Verification")
for name in ["databases", "raw_input", "results", "reports"]:
    lpath  = os.path.join(REPO_DIR, name)
    is_lnk = os.path.islink(lpath)
    is_dir = os.path.isdir(lpath)
    target = os.readlink(lpath) if is_lnk else "(not a symlink)"
    drive_ok = DRIVE_ROOT in target if is_lnk else False
    _check("Symlinks", name, is_lnk and is_dir and drive_ok,
           detail=f"→ {target}")

# ── 4. Database Verification ──────────────────────────────────────────────────
# Database paths as hardcoded in the actual wrapper scripts:
#   run_checkv.sh   → $ROOT/databases/checkv-db
#   run_pharokka.sh → $ROOT/databases/pharokka_db
#   run_phold.sh    → $ROOT/databases/phold_db
#   run_phynteny.sh → $ROOT/databases/phynteny_models/models
#   run_acrdb_blast.sh (+ config.yaml) → databases/acrdb_db/122_KnownAcr/Known_Acr.faa
_section("4. Database Verification")
DB_DIR = os.path.join(REPO_DIR, "databases")
DB_CHECKS = [
    ("CheckV DB",       "checkv-db",                             False, "run_checkv.sh: DB"),
    ("Pharokka DB",     "pharokka_db",                           False, "run_pharokka.sh: DB"),
    ("Phold DB",        "phold_db",                              False, "run_phold.sh: DB_PATH"),
    ("Phynteny models", "phynteny_models/models",                False, "run_phynteny.sh: MODELS_PATH"),
    ("AcrDB FASTA",     "acrdb_db/122_KnownAcr/Known_Acr.faa",  True,  "run_acrdb_blast.sh: DB / config.yaml: acrdb_db"),
]
db_missing = []
for display, rel, is_file, script_var in DB_CHECKS:
    full   = os.path.join(DB_DIR, rel)
    exists = os.path.isfile(full) if is_file else os.path.isdir(full)
    _check("Databases", display, exists,
           detail=f"databases/{rel}  [{script_var}]")
    if not exists:
        db_missing.append((display, rel, is_file, script_var))

if db_missing:
    print("\n" + "!"*62)
    print("  MISSING DATABASES — Download instructions")
    print("!"*62)
    for display, rel, is_file, _ in db_missing:
        dst = f"{DRIVE_ROOT_PATH}/databases/{rel}"
        print(f"\n  [{display}]")
        print(f"  Expected: {dst}")
        if "checkv" in rel:
            print(f"  Download:\n    conda run -n env_checkv checkv database-update -d '{dst}'")
        elif "pharokka" in rel:
            print(f"  Download:\n    conda run -n env_pharokka install_databases.py -o '{dst}'")
        elif "phold" in rel:
            print(f"  Download:\n    conda run -n pholdENV phold install-database -d '{dst}'")
        elif "phynteny" in rel:
            print("  Download: See phynteny_transformer documentation for model download.")
            print(f"  Place models in: {DRIVE_ROOT_PATH}/databases/phynteny_models/models/")
        elif "acrdb" in rel:
            print("  Download: AcrDB FASTA available at https://doi.org/10.1093/nar/gkaa009")
            print(f"  Place at: {DRIVE_ROOT_PATH}/databases/acrdb_db/122_KnownAcr/Known_Acr.faa")
            print(f"  Then index: conda run -n env_acrdb makeblastdb -in '{dst}' -dbtype prot")
    print("\n  Use Section 5b (optional cell) to trigger downloads.")
    print("!"*62)

# ── 5. Input FASTA Validation ─────────────────────────────────────────────────
_section("5. Input FASTA Files")
_sample_list   = [s.strip() for s in SAMPLES.split(",") if s.strip()]
RAW_INPUT_DIR  = os.path.join(REPO_DIR, "raw_input")
fasta_missing  = []
for sample in _sample_list:
    fasta = os.path.join(RAW_INPUT_DIR, f"{sample}.fasta")
    if not os.path.exists(fasta):
        _check("Input", f"{sample}.fasta", False,
               detail=f"Not found in raw_input/")
        fasta_missing.append(sample)
        continue
    try:
        with open(fasta, "r") as fh:
            first = fh.read(1)
        valid  = first == ">"
        kb     = os.path.getsize(fasta) // 1024
        _check("Input", f"{sample}.fasta", valid,
               detail=f"{kb} KB" + ("" if valid else " — invalid FASTA (no '>' header)"))
        if not valid:
            fasta_missing.append(sample)
    except Exception as e:
        _check("Input", f"{sample}.fasta", False, detail=str(e))
        fasta_missing.append(sample)

if fasta_missing:
    print(f"\n  Upload FASTAs to Google Drive: {DRIVE_ROOT_PATH}/raw_input/")
    print("  Or use Cell 25 (Section 9 — Upload FASTA files) to upload from your computer.")

# ── 6. Conda Environment Check ────────────────────────────────────────────────
_section("6. Conda Environments")
ENV_TABLE = [
    ("env_checkv",   "CheckV"),
    ("env_pharokka", "Pharokka"),
    ("pholdENV",     "PHOLD"),
    ("phyntenyENV",  "Phynteny"),
    ("env_phage_ml", "PhageDPO + PhageRBPdetect v4 + PhageTailFinder"),
    ("phage",        "Post-processing scripts (build_annotated_gb, visualize_genome...)"),
    ("env_bacphlip", "BACPHLIP"),
    ("env_rgi",      "RGI"),
    ("env_abricate", "Abricate (CARD + VFDB)"),
    ("env_acrdb",    "AcrDB BLASTP"),
]
for env_name, used_by in ENV_TABLE:
    _check("Conda", env_name, _conda_env_exists(env_name),
           detail=f"Used by: {used_by}")

# ── 7. Tool Executables ───────────────────────────────────────────────────────
_section("7. Tool Executables")
for exe, desc in [
    ("snakemake", "Workflow orchestrator"),
    ("conda",     "Conda package manager"),
    ("mamba",     "Mamba fast solver"),
    ("git",       "Version control"),
    ("flock",     "Concurrency lock used by run_phagerbpdetect_v4.sh"),
]:
    _check("Exe", exe, shutil.which(exe) is not None, detail=desc)

# ── 8. Script Permissions ─────────────────────────────────────────────────────
_section("8. Script Permissions")
for rel in [
    "scripts/checkv/run_checkv.sh",
    "scripts/pharokka/run_pharokka.sh",
    "scripts/phage/run_phold.sh",
    "scripts/phage/run_phynteny.sh",
    "scripts/bacphlip/run_bacphlip.sh",
    "scripts/rgi/run_rgi.sh",
    "scripts/abricate/run_abricate.sh",
    "scripts/acrdb/run_acrdb_blast.sh",
]:
    full = os.path.join(REPO_DIR, rel)
    if os.path.exists(full):
        is_exec = os.access(full, os.X_OK)
        _check("Perms", rel, is_exec, detail="executable" if is_exec else "NOT executable")
    else:
        _check("Perms", rel, False, detail="file not found")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  VALIDATION SUMMARY")
print("=" * 62)
by_status = {"PASS": [], "WARN": [], "FAIL": []}
for cat, name, status, detail in checks:
    by_status[status].append((cat, name, detail))
n_pass = len(by_status["PASS"])
n_warn = len(by_status["WARN"])
n_fail = len(by_status["FAIL"])
print(f"   PASS : {n_pass}")
print(f"   WARN : {n_warn}")
print(f"   FAIL : {n_fail}")

if n_fail > 0:
    print("\n  ❌ BLOCKING FAILURES — fix before running:")
    for cat, name, detail in by_status["FAIL"]:
        print(f"     • [{cat}] {name}" + (f" — {detail}" if detail else ""))
elif n_warn > 0:
    print("\n  ⚠️  Warnings (non-blocking):")
    for cat, name, detail in by_status["WARN"]:
        print(f"     • [{cat}] {name}" + (f" — {detail}" if detail else ""))
    print("\n  ✅ No blocking failures. Pipeline is READY TO RUN (with caveats above).")
else:
    print("\n  ✅ All checks passed. Pipeline is READY TO RUN.")


In [ ]:
# ==============================================================================
# CELL 11 — [OPTIONAL] Database Download Helper
# ==============================================================================
# Run this cell ONLY if databases are missing on Google Drive.
# Downloads go directly to Drive and persist across sessions.
# ==============================================================================

DOWNLOAD_CHECKV   = False   # Set True → download CheckV database
DOWNLOAD_PHAROKKA = False   # Set True → download Pharokka databases
DOWNLOAD_PHOLD    = False   # Set True → download Phold database
# Phynteny models and AcrDB must be obtained manually (see validation output above)

import subprocess, os

DRIVE_ROOT_PATH = f"/content/drive/MyDrive/{DRIVE_ROOT}"
CONDA_BIN       = "/opt/conda/bin/conda"
DB_DRIVE        = os.path.join(DRIVE_ROOT_PATH, "databases")

def _download(label, cmd):
    print(f"\n[{label}] Running: {cmd}")
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout
    print(out[-3000:] if len(out) > 3000 else out)
    if r.returncode == 0:
        print(f"✅ [{label}] Done.")
    else:
        print(f"❌ [{label}] Failed (exit {r.returncode}).")

if DOWNLOAD_CHECKV:
    dst = os.path.join(DB_DRIVE, "checkv-db")
    os.makedirs(dst, exist_ok=True)
    _download("CheckV DB",
              f"{CONDA_BIN} run -n env_checkv checkv database-update -d '{dst}'")

if DOWNLOAD_PHAROKKA:
    dst = os.path.join(DB_DRIVE, "pharokka_db")
    os.makedirs(dst, exist_ok=True)
    _download("Pharokka DB",
              f"{CONDA_BIN} run -n env_pharokka install_databases.py -o '{dst}'")

if DOWNLOAD_PHOLD:
    dst = os.path.join(DB_DRIVE, "phold_db")
    os.makedirs(dst, exist_ok=True)
    _download("Phold DB",
              f"{CONDA_BIN} run -n pholdENV phold install-database -d '{dst}'")

if not any([DOWNLOAD_CHECKV, DOWNLOAD_PHAROKKA, DOWNLOAD_PHOLD]):
    print("No downloads requested. Set the flags above to True to download specific databases.")


---
## Section 6 — Pipeline Configuration

Resolves runtime parameters and builds the Snakemake command. Nothing is executed here.


In [ ]:
# ==============================================================================
# CELL 12 — Prepare Run Configuration
# ==============================================================================
import os
from datetime import date

REPO_DIR = "/content/PhagePipeline"

# ── Resolve RUN_ID ────────────────────────────────────────────────────────────
RESOLVED_RUN_ID = RUN_ID.strip() if RUN_ID and RUN_ID.strip() else date.today().strftime("%Y-%m-%d")
print(f"Run ID          : {RESOLVED_RUN_ID}")

# ── Resolve sample list ───────────────────────────────────────────────────────
SAMPLE_LIST = [s.strip() for s in SAMPLES.split(",") if s.strip()]
SAMPLES_CSV = ",".join(SAMPLE_LIST)
print(f"Samples ({len(SAMPLE_LIST)})    : {SAMPLES_CSV}")

RAW_INPUT_DIR = os.path.join(REPO_DIR, "raw_input")
print("\nFASTA status:")
for s in SAMPLE_LIST:
    fasta = os.path.join(RAW_INPUT_DIR, f"{s}.fasta")
    icon  = "✅" if os.path.exists(fasta) else "❌ MISSING"
    print(f"   {icon}  {s}.fasta")

# ── Output paths ──────────────────────────────────────────────────────────────
REPORTS_RUN_DIR = os.path.join(REPO_DIR, "reports", RESOLVED_RUN_ID)
RESULTS_DIR     = os.path.join(REPO_DIR, "results")

# ── Build Snakemake command ───────────────────────────────────────────────────
# IMPORTANT: This pipeline does NOT use Snakemake's --use-conda mechanism.
# Environment activation is handled INTERNALLY by the wrapper scripts themselves:
#   run_checkv.sh      → conda run -n env_checkv ...
#   run_pharokka.sh    → conda run -n env_pharokka ...
#   run_phold.sh       → conda run -n pholdENV ...
#   run_phynteny.sh    → conda run -n phyntenyENV ...
#   run_bacphlip.sh    → conda run -n env_bacphlip ...
#   run_rgi.sh         → conda run -n env_rgi ...
#   run_abricate.sh    → conda run -n env_abricate ...
#   run_acrdb_blast.sh → conda run -n env_acrdb ...
#   Snakefile rules    → conda run -n phage python ...
#   run_phagedpo.sh / run_phagerbpdetect_v4.sh / run_phagetailfinder.sh → conda run -n env_phage_ml ...
# We only need /opt/conda/bin on PATH so the wrappers can find the 'conda' binary.
import os as _os
_CONDA_PATH = "/opt/conda"
_os.environ["PATH"] = f"{_CONDA_PATH}/bin:" + _os.environ.get("PATH", "")
_os.environ["CONDA_EXE"] = f"{_CONDA_PATH}/bin/conda"

SNAKEMAKE_CMD = (
    f"snakemake"
    f" --snakefile '{REPO_DIR}/Snakefile'"
    f" --cores {THREADS}"
    f" --config samples='{SAMPLES_CSV}' run_id='{RESOLVED_RUN_ID}' threads={THREADS}"
    f" --rerun-incomplete"
    f" --printshellcmds"
)
if SNAKEMAKE_EXTRA_FLAGS.strip():
    SNAKEMAKE_CMD += f" {SNAKEMAKE_EXTRA_FLAGS.strip()}"

print(f"\nSnakemake command:")
print(f"  {SNAKEMAKE_CMD}")
print(f"\nConda binary on PATH: {_os.environ['PATH'].split(':')[0]}/conda")
print(f"\nOutput locations:")
print(f"  Reports : {REPORTS_RUN_DIR}/")
print(f"  Results : {RESULTS_DIR}/")
print(f"  Scores  : {REPORTS_RUN_DIR}/phage_scores.tsv")


---
## Section 7 — Pipeline Execution

Executes the Snakemake workflow. **The pipeline is run exactly as implemented — no rules are rewritten, no tools are replaced.**

### Runtime estimates (Colab free tier, 4 cores, per sample)

| Tool | CPU time | GPU time |
|------|----------|----------|
| CheckV | ~2–5 min | — |
| Pharokka | ~5–15 min | — |
| PHOLD | ~15–30 min | ~3–8 min |
| Phynteny | ~5–15 min | — |
| PhageDPO | ~1–3 min | — |
| PhageRBPdetect v4 | ~10–25 min | ~3–8 min |
| PhageTailFinder | ~2–5 min | — |
| BACPHLIP | ~2–5 min | — |
| RGI | ~3–10 min | — |
| Abricate ×2 | ~1–3 min | — |
| AcrDB BLAST | ~1–2 min | — |
| Post-processing | ~2–5 min | — |
| **Total** | **~50–120 min** | **~25–70 min** |


In [ ]:
# ==============================================================================
# CELL 13 — Dry Run (preview execution plan)
# ==============================================================================
import subprocess

REPO_DIR = "/content/PhagePipeline"

print("Running Snakemake --dry-run to preview execution plan...")
print("=" * 62)

r = subprocess.run(
    SNAKEMAKE_CMD + " --dry-run --quiet",
    shell=True, text=True, cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(r.stdout)

if r.returncode != 0:
    print(f"⚠️  Dry-run exited with code {r.returncode}")
    print("This may indicate missing inputs or configuration problems.")
    print("Review the output above before proceeding to the Execute cell.")
else:
    print("✅ Dry-run complete. Execution plan looks valid. Proceed to next cell.")


In [ ]:
# ==============================================================================
# CELL 14 — Execute Pipeline
# ==============================================================================
# Streams Snakemake output in real time.
# DO NOT interrupt — let it run to completion.
# If a rule fails, use the Log Viewer (Cell 23) to inspect errors.
# ==============================================================================
import subprocess
from datetime import datetime

REPO_DIR = "/content/PhagePipeline"

t_start = datetime.now()
print("=" * 62)
print("  PhagePipeline — Execution Start")
print(f"  Samples  : {SAMPLES_CSV}")
print(f"  Run ID   : {RESOLVED_RUN_ID}")
print(f"  Threads  : {THREADS}")
print(f"  Started  : {t_start.strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 62)

proc = subprocess.Popen(
    SNAKEMAKE_CMD,
    shell=True, text=True, cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, universal_newlines=True
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
t_end      = datetime.now()
duration   = t_end - t_start
dur_str    = str(duration).split(".")[0]

print("\n" + "=" * 62)
PIPELINE_EXIT_CODE = proc.returncode
if proc.returncode == 0:
    print(f"  ✅ PIPELINE COMPLETE")
    print(f"  Duration : {dur_str}")
    print(f"  Reports  : {REPORTS_RUN_DIR}/")
    print(f"  Scores   : {REPORTS_RUN_DIR}/phage_scores.tsv")
else:
    print(f"  ❌ PIPELINE FAILED (exit code {proc.returncode})")
    print(f"  Duration : {dur_str}")
    print(f"  Check logs in: {RESULTS_DIR}/*/logs/")
    print("  Use the Log Viewer (Cell 23) to inspect errors.")
print("=" * 62)


---
## Section 8 — Results & Reporting

Displays all pipeline outputs. Run any cell individually to inspect a specific output.


In [ ]:
# ==============================================================================
# CELL 15 — Phage Candidate Scores Table
# Reads: reports/{run_id}/phage_scores.tsv
# Fields: sample, status, total_score, quality_score, lifestyle_score,
#         safety_score, lysis_score, host_score, rejection_reasons + notes
# Produced by: scripts/score_and_rank.py
# ==============================================================================
import os, pandas as pd
from IPython.display import display, HTML

REPO_DIR        = "/content/PhagePipeline"
REPORTS_RUN_DIR = os.path.join(REPO_DIR, "reports", RESOLVED_RUN_ID)
SCORES_TSV      = os.path.join(REPORTS_RUN_DIR, "phage_scores.tsv")

if not os.path.exists(SCORES_TSV):
    print(f"⚠️  Scores file not found: {SCORES_TSV}")
    print("Run the pipeline first (Cell 14).")
else:
    df = pd.read_csv(SCORES_TSV, sep="\t")

    STATUS_STYLE = {
        "PASS":     "background:#d4edda; color:#1a5c1a; font-weight:bold; text-align:center",
        "REVIEW":   "background:#fff3cd; color:#856404; font-weight:bold; text-align:center",
        "FAIL":     "background:#f8d7da; color:#721c24; font-weight:bold; text-align:center",
        "REJECTED": "background:#f5c6cb; color:#491217; font-weight:bold; text-align:center",
    }

    def _style_status(v):  return STATUS_STYLE.get(str(v), "")
    def _style_total(v):
        try:
            x = float(v)
            if x >= 80: return "background:#d4edda; font-weight:bold"
            if x >= 50: return "background:#fff3cd"
            return "background:#f8d7da"
        except: return ""

    SHOW_COLS = [c for c in [
        "sample", "status", "total_score",
        "quality_score", "lifestyle_score", "safety_score",
        "lysis_score", "host_score", "rejection_reasons"
    ] if c in df.columns]

    styled = (
        df[SHOW_COLS].style
        .applymap(_style_status, subset=["status"])
        .applymap(_style_total,  subset=["total_score"])
        .set_caption(f"Phage Candidate Scores — Run: {RESOLVED_RUN_ID}")
        .set_table_styles([{"selector": "caption",
                            "props": [("font-size","14px"),("font-weight","bold"),("padding","8px")]}])
        .format(precision=1, na_rep="—")
    )
    display(styled)

    # Scoring legend
    display(HTML("""
    <table style='font-size:12px; border-collapse:collapse; margin-top:12px'>
      <tr><th style='text-align:left; padding:4px 12px'>Dimension</th><th>Max</th><th style='text-align:left; padding:4px 12px'>Source</th></tr>
      <tr><td style='padding:4px 12px'>Quality</td><td>20</td><td style='padding:4px 12px'>CheckV genome quality + completeness</td></tr>
      <tr><td>Lifestyle</td><td>30</td><td>BACPHLIP × annotation markers (integrase/recombinase/transposase)</td></tr>
      <tr><td>Safety</td><td>30</td><td>RGI + Abricate-CARD (AMR) + Abricate-VFDB (virulence) + AcrDB (anti-CRISPR)</td></tr>
      <tr><td>Lysis</td><td>10</td><td>Named lysis proteins (holin, endolysin, lysin, spanin)</td></tr>
      <tr><td>Host</td><td>10</td><td>Tail, head/packaging, and connector structural proteins</td></tr>
    </table>
    <p style='font-size:12px; margin-top:8px'>
    <b>Thresholds:</b> PASS ≥ 80 | REVIEW 50–79 | FAIL &lt; 50 | REJECTED (hard disqualifier: TEMPERATE, AMR, or virulence confirmed)
    </p>
    """))


In [ ]:
# ==============================================================================
# CELL 16 — Functional Genome Maps
# Reads: reports/{run_id}/{sample}/{sample}_genome_map.png
# Produced by: scripts/visualize_genome.py (pygenomeviz)
# ==============================================================================
import os
from IPython.display import Image, display, HTML

REPO_DIR        = "/content/PhagePipeline"
REPORTS_RUN_DIR = os.path.join(REPO_DIR, "reports", RESOLVED_RUN_ID)
SAMPLE_LIST     = [s.strip() for s in SAMPLES.split(",") if s.strip()]

print(f"Functional Genome Maps — Run: {RESOLVED_RUN_ID}")

any_found = False
for sample in SAMPLE_LIST:
    png = os.path.join(REPORTS_RUN_DIR, sample, f"{sample}_genome_map.png")
    svg = os.path.join(REPORTS_RUN_DIR, sample, f"{sample}_genome_map.svg")
    if os.path.exists(png):
        any_found = True
        display(HTML(f"<h3 style='margin-top:24px'>🧬 {sample}</h3>"))
        display(Image(filename=png, width=950))
        svg_note = f" | SVG: <a href='{svg}'>{svg}</a>" if os.path.exists(svg) else ""
        display(HTML(f"<small>PNG: {png}{svg_note}</small>"))
    else:
        print(f"⚠️  Genome map not found for {sample}: {png}")

if not any_found:
    print("No genome maps found. The pipeline may not have completed post-processing.")


In [ ]:
# ==============================================================================
# CELL 17 — Protein Annotation Tables
# Reads: reports/{run_id}/{sample}/{sample}_protein_table.tsv
# Produced by: scripts/build_protein_table.py
# Columns: protein_id, start, end, strand, product, category, confidence,
#          evidence_source (PHOLD | PHOLD+PHYNTENY | PHYNTENY | none)
# ==============================================================================
import os, pandas as pd
from IPython.display import display, HTML

REPO_DIR        = "/content/PhagePipeline"
REPORTS_RUN_DIR = os.path.join(REPO_DIR, "reports", RESOLVED_RUN_ID)
SAMPLE_LIST     = [s.strip() for s in SAMPLES.split(",") if s.strip()]

# Category colours (matches visualize_genome.py CATEGORY_COLORS)
CAT_COLORS = {
    "head and packaging":                                  "#4E79A7",
    "tail":                                                "#F28E2B",
    "connector":                                           "#E15759",
    "lysis":                                               "#76B7B2",
    "DNA, RNA and nucleotide metabolism":                  "#59A14F",
    "transcription regulation":                            "#EDC948",
    "integration and excision":                            "#B07AA1",
    "moron, auxiliary metabolic gene and host takeover":   "#FF9DA7",
    "other":                                               "#9C755F",
    "unknown function":                                    "#BAB0AC",
}

def _lighten(hex_c, factor=0.35):
    h = hex_c.lstrip("#")
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    lr = int(r + (255-r)*factor); lg = int(g + (255-g)*factor); lb = int(b + (255-b)*factor)
    return f"#{lr:02X}{lg:02X}{lb:02X}"

def _style_row(row):
    bg = _lighten(CAT_COLORS.get(str(row.get("category","")), "#D3D3D3"))
    return [f"background-color:{bg}"]*len(row)

def _style_conf(v):
    return {"high":"font-weight:bold;color:#1a5c1a",
            "medium":"color:#856404",
            "low":"color:#721c24",
            "none":"color:#6c757d;font-style:italic"}.get(str(v).lower(),"")

for sample in SAMPLE_LIST:
    tsv = os.path.join(REPORTS_RUN_DIR, sample, f"{sample}_protein_table.tsv")
    if not os.path.exists(tsv):
        print(f"⚠️  Protein table not found for {sample}")
        continue
    df = pd.read_csv(tsv, sep="\t")
    display(HTML(f"<h3>🧬 {sample} — Protein Annotation ({len(df)} CDS)</h3>"))

    styled = (
        df.style
        .apply(_style_row, axis=1)
        .applymap(_style_conf, subset=["confidence"] if "confidence" in df.columns else [])
        .set_properties(**{"font-size":"12px", "white-space":"nowrap"})
        .format(na_rep="—")
    )
    display(styled)

    if "category" in df.columns:
        cc = df["category"].value_counts()
        total = len(df)
        print(f"\nCategory distribution ({sample}):")
        for cat, cnt in cc.items():
            pct = cnt / total * 100
            bar = "█" * int(pct / 2)
            print(f"  {cat:<48} {cnt:>4}  ({pct:5.1f}%)  {bar}")
    print()


In [ ]:
# ==============================================================================
# CELL 18 — Tool Summary Tables
# Reads: reports/{run_id}/{sample}/{sample}_tool_summary.tsv
# Produced by: scripts/summarize_sample.py
# Columns: sample, tool, metric, value, detail, flag
# Flags: DISQUALIFY | ERROR | WARN | (empty)
# ==============================================================================
import os, pandas as pd
from IPython.display import display, HTML

REPO_DIR        = "/content/PhagePipeline"
REPORTS_RUN_DIR = os.path.join(REPO_DIR, "reports", RESOLVED_RUN_ID)
SAMPLE_LIST     = [s.strip() for s in SAMPLES.split(",") if s.strip()]

FLAG_STYLE = {
    "DISQUALIFY": "background:#f5c6cb; color:#491217; font-weight:bold",
    "ERROR":      "background:#f8d7da; color:#721c24; font-weight:bold",
    "WARN":       "background:#fff3cd; color:#856404",
}

def _style_flag(v): return FLAG_STYLE.get(str(v), "")

for sample in SAMPLE_LIST:
    tsv = os.path.join(REPORTS_RUN_DIR, sample, f"{sample}_tool_summary.tsv")
    if not os.path.exists(tsv):
        print(f"⚠️  Tool summary not found for {sample}")
        continue
    df = pd.read_csv(tsv, sep="\t")
    display(HTML(f"<h3>🔬 {sample} — Tool Summary ({len(df)} metrics)</h3>"))

    styled = (
        df.style
        .applymap(_style_flag, subset=["flag"] if "flag" in df.columns else [])
        .set_properties(**{"font-size":"12px", "text-align":"left"})
        .format(na_rep="—")
    )
    display(styled)

    if "flag" in df.columns:
        flagged = df[df["flag"].isin(["DISQUALIFY","ERROR","WARN"])]
        if len(flagged):
            print(f"\nFlags for {sample}:")
            for _, row in flagged.iterrows():
                icon = {"DISQUALIFY":"❌","ERROR":"🔴","WARN":"⚠️ "}.get(row["flag"],"")
                print(f"  {icon} [{row['flag']}] {row['tool']} / {row['metric']} = {row['value']}")
                if str(row.get("detail","")).strip():
                    print(f"         {row['detail']}")
        else:
            print(f"  ✅ No flags for {sample}")
    print()


In [ ]:
# ==============================================================================
# CELL 19 — Human-Readable Candidate Reports
# Reads: reports/{run_id}/{sample}/{sample}_report.txt
# Produced by: scripts/summarize_sample.py
# Sections: GENOME | ANNOTATION | LIFESTYLE | SAFETY: AMR | SAFETY: VIRULENCE |
#           ANTI-CRISPR | LYSIS MACHINERY
# ==============================================================================
import os
from IPython.display import display, HTML

REPO_DIR        = "/content/PhagePipeline"
REPORTS_RUN_DIR = os.path.join(REPO_DIR, "reports", RESOLVED_RUN_ID)
SAMPLE_LIST     = [s.strip() for s in SAMPLES.split(",") if s.strip()]

for sample in SAMPLE_LIST:
    rpt = os.path.join(REPORTS_RUN_DIR, sample, f"{sample}_report.txt")
    if not os.path.exists(rpt):
        print(f"⚠️  Report not found: {rpt}")
        continue
    with open(rpt) as fh:
        content = fh.read()
    # Highlight DISQUALIFY/WARN lines
    content_html = content.replace("&","&amp;").replace("<","&lt;")
    display(HTML(
        f"<h3>📋 {sample} — Candidate Report</h3>"
        f"<pre style='background:#f8f9fa;border:1px solid #dee2e6;border-radius:6px;"
        f"padding:16px;font-size:12px;font-family:Consolas,monospace;white-space:pre-wrap;'>"
        f"{content_html}</pre>"
    ))


In [ ]:
# ==============================================================================
# CELL 20 — PhageRBPdetect v4 — Receptor-Binding Protein Predictions
# Reads: results/{sample}/phagerbpdetect/{sample}_rbp_predictions.tsv
#        results/{sample}/phagerbpdetect/{sample}_rbp_summary.tsv
# Produced by: scripts/phagerbpdetect/run_phagerbpdetect_v4.sh
# Columns: protein_id, rbp_prediction (0/1), rbp_score, rbp_label (RBP/non-RBP)
# ==============================================================================
import os, pandas as pd
from IPython.display import display, HTML

REPO_DIR        = "/content/PhagePipeline"
RESULTS_DIR     = os.path.join(REPO_DIR, "results")
SAMPLE_LIST     = [s.strip() for s in SAMPLES.split(",") if s.strip()]

print("PhageRBPdetect v4 — Receptor-Binding Protein Predictions")
print("=" * 62)

for sample in SAMPLE_LIST:
    pred_tsv = os.path.join(RESULTS_DIR, sample, "phagerbpdetect",
                             f"{sample}_rbp_predictions.tsv")
    summ_tsv = os.path.join(RESULTS_DIR, sample, "phagerbpdetect",
                             f"{sample}_rbp_summary.tsv")

    if not os.path.exists(pred_tsv):
        print(f"⚠️  RBP predictions not found for {sample}")
        continue

    if os.path.exists(summ_tsv):
        summ = pd.read_csv(summ_tsv, sep="\t")
        display(HTML(f"<h4>📊 {sample} — RBP Summary</h4>"))
        display(summ.style.format(precision=4, na_rep="—"))

    df   = pd.read_csv(pred_tsv, sep="\t")
    label_col = "rbp_label" if "rbp_label" in df.columns else None
    pred_col  = "rbp_prediction" if "rbp_prediction" in df.columns else None

    if label_col:
        rbps = df[df[label_col] == "RBP"]
    elif pred_col:
        rbps = df[df[pred_col] == 1]
    else:
        rbps = df

    display(HTML(f"<h4>🎯 {sample} — RBP Candidates ({len(rbps)} / {len(df)} proteins)</h4>"))
    if len(rbps):
        display(
            rbps.style
            .applymap(lambda v: "background:#d4edda;font-weight:bold" if v=="RBP" else "",
                       subset=[label_col] if label_col else [])
            .format(precision=4, na_rep="—")
        )
    else:
        print(f"   {sample}: No RBPs predicted.")
    print()


In [ ]:
# ==============================================================================
# CELL 21 — PhageTailFinder — Tail Protein Classifications
# Reads: results/{sample}/phagetailfinder/{sample}_tailfinder.tsv
# Produced by: scripts/phagetailfinder/run_phagetailfinder.sh
# Columns: phage_content, phage_id, protein_id, protein_size, protein_count,
#          start_index, end_index, protein_def, tail_or_not, sample, is_tail
# ==============================================================================
import os, pandas as pd
from IPython.display import display, HTML

REPO_DIR    = "/content/PhagePipeline"
RESULTS_DIR = os.path.join(REPO_DIR, "results")
SAMPLE_LIST = [s.strip() for s in SAMPLES.split(",") if s.strip()]

print("PhageTailFinder — Tail Protein Classification")
print("=" * 62)

for sample in SAMPLE_LIST:
    tsv = os.path.join(RESULTS_DIR, sample, "phagetailfinder",
                        f"{sample}_tailfinder.tsv")
    if not os.path.exists(tsv):
        print(f"⚠️  TailFinder results not found for {sample}")
        continue
    df = pd.read_csv(tsv, sep="\t")
    n_total = len(df)
    tail_col = "tail_or_not" if "tail_or_not" in df.columns else "is_tail"
    n_tail = int((df[tail_col] == 1).sum()) if tail_col in df.columns else 0

    display(HTML(f"<h4>🦠 {sample} — TailFinder ({n_tail}/{n_total} tail proteins)</h4>"))
    tail_df = df[df[tail_col] == 1] if tail_col in df.columns and n_tail else df
    if n_tail:
        display(
            tail_df.style
            .applymap(lambda v: "background:#d4edda" if v==1 else "",
                       subset=[tail_col] if tail_col in tail_df.columns else [])
            .format(na_rep="—")
        )
    else:
        print(f"   {sample}: No tail proteins classified.")
    print()


In [ ]:
# ==============================================================================
# CELL 22 — PhageDPO — Depolymerase Predictions
# Reads: results/{sample}/phagedpo/{sample}_cds_output.tsv
#        results/{sample}/phagedpo/{sample}_cds_output.html
# Produced by: scripts/phagedpo/run_phagedpo.sh
# Note: The TSV is converted from the HTML output by phagedpo_cli.py
# ==============================================================================
import os, pandas as pd
from IPython.display import display, HTML

REPO_DIR    = "/content/PhagePipeline"
RESULTS_DIR = os.path.join(REPO_DIR, "results")
SAMPLE_LIST = [s.strip() for s in SAMPLES.split(",") if s.strip()]

print("PhageDPO — Depolymerase Predictions")
print("=" * 62)

for sample in SAMPLE_LIST:
    tsv  = os.path.join(RESULTS_DIR, sample, "phagedpo", f"{sample}_cds_output.tsv")
    html = os.path.join(RESULTS_DIR, sample, "phagedpo", f"{sample}_cds_output.html")

    if not os.path.exists(tsv):
        print(f"⚠️  PhageDPO results not found for {sample}")
        continue

    df = pd.read_csv(tsv, sep="\t")
    display(HTML(f"<h4>🧪 {sample} — PhageDPO ({len(df)} CDS)</h4>"))

    score_col = next((c for c in df.columns if "score" in c.lower() or "pred" in c.lower()), None)
    if score_col:
        styled = df.style.background_gradient(subset=[score_col], cmap="RdYlGn").format(precision=4, na_rep="—")
    else:
        styled = df.style.format(na_rep="—")
    display(styled)

    if os.path.exists(html):
        display(HTML(f"<small>Full HTML report: {html}</small>"))
    print()


In [ ]:
# ==============================================================================
# CELL 23 — Log Viewer (error diagnosis)
# Reads: results/{sample}/logs/*.log
# Shows last N lines of each tool log. Use after a pipeline failure.
# ==============================================================================
import os
from IPython.display import display, HTML

REPO_DIR        = "/content/PhagePipeline"
RESULTS_DIR     = os.path.join(REPO_DIR, "results")
SAMPLE_LIST     = [s.strip() for s in SAMPLES.split(",") if s.strip()]
LOG_TAIL_LINES  = 40

# All log files as named by the Snakemake log: directives
TOOL_LOGS = [
    "checkv.log", "pharokka.log", "phold.log", "phynteny.log",
    "phagedpo.log", "phagerbpdetect.log", "phagetailfinder.log",
    "bacphlip.log", "rgi.log", "abricate_card.log", "abricate_vfdb.log",
    "acrdb.log", "build_annotated_gb.log", "build_protein_table.log",
    "visualize_genome.log", "summarize_sample.log",
]

print("=" * 62)
print("  Log Viewer")
print("=" * 62)

for sample in SAMPLE_LIST:
    logs_dir = os.path.join(RESULTS_DIR, sample, "logs")
    display(HTML(f"<h4>📂 {sample}</h4>"))
    if not os.path.exists(logs_dir):
        print(f"   Logs directory not found: {logs_dir}")
        continue
    found_any = False
    for lf in TOOL_LOGS:
        lpath = os.path.join(logs_dir, lf)
        if not os.path.exists(lpath):
            continue
        found_any = True
        sz = os.path.getsize(lpath)
        print(f"\n  {'─'*50}")
        print(f"  {lf}  ({sz:,} bytes)")
        print(f"  {'─'*50}")
        with open(lpath, errors="replace") as fh:
            lines = fh.readlines()
        if len(lines) > LOG_TAIL_LINES:
            print(f"  [...{len(lines)-LOG_TAIL_LINES} earlier lines omitted...]")
        print("".join(lines[-LOG_TAIL_LINES:]), end="")
    if not found_any:
        print("   No log files found (pipeline may not have started for this sample).")

# score_and_rank log (lives in results/logs/ not results/{sample}/logs/)
rank_log = os.path.join(RESULTS_DIR, "logs", f"{RESOLVED_RUN_ID}_score_and_rank.log")
if os.path.exists(rank_log):
    print(f"\n  {'─'*50}")
    print(f"  score_and_rank.log")
    print(f"  {'─'*50}")
    with open(rank_log, errors="replace") as fh:
        print(fh.read())


In [ ]:
# ==============================================================================
# CELL 24 — Output Manifest
# Lists all expected output files with their status and sizes.
# All files persist on Google Drive via symlinks.
# ==============================================================================
import os, pandas as pd
from IPython.display import display, HTML

REPO_DIR        = "/content/PhagePipeline"
RESULTS_DIR     = os.path.join(REPO_DIR, "results")
REPORTS_RUN_DIR = os.path.join(REPO_DIR, "reports", RESOLVED_RUN_ID)
SAMPLE_LIST     = [s.strip() for s in SAMPLES.split(",") if s.strip()]

manifest = []

def _add(sample, fname, location, desc, path):
    exists  = os.path.exists(path)
    size_kb = os.path.getsize(path)//1024 if exists else None
    manifest.append({
        "sample":      sample,
        "file":        fname,
        "location":    location,
        "description": desc,
        "size_KB":     size_kb,
        "status":      "✅" if exists else "❌",
    })

# Report files (reports/{run_id}/{sample}/)
for sample in SAMPLE_LIST:
    rd = os.path.join(REPORTS_RUN_DIR, sample)
    loc = f"reports/{RESOLVED_RUN_ID}/{sample}/"
    _add(sample, f"{sample}_annotated.gbk",    loc, "Enriched GenBank — PHOLD+Phynteny", os.path.join(rd, f"{sample}_annotated.gbk"))
    _add(sample, f"{sample}_protein_table.tsv",loc, "One-row-per-CDS protein table",     os.path.join(rd, f"{sample}_protein_table.tsv"))
    _add(sample, f"{sample}_genome_map.png",   loc, "Functional genome map (PNG)",       os.path.join(rd, f"{sample}_genome_map.png"))
    _add(sample, f"{sample}_genome_map.svg",   loc, "Functional genome map (SVG)",       os.path.join(rd, f"{sample}_genome_map.svg"))
    _add(sample, f"{sample}_tool_summary.tsv", loc, "All tool metrics and flags",        os.path.join(rd, f"{sample}_tool_summary.tsv"))
    _add(sample, f"{sample}_report.txt",       loc, "Human-readable candidate report",   os.path.join(rd, f"{sample}_report.txt"))

# Scores file
_add("(all)", "phage_scores.tsv", f"reports/{RESOLVED_RUN_ID}/",
     "Cross-sample scoring table", os.path.join(REPORTS_RUN_DIR, "phage_scores.tsv"))

# Result files (results/{sample}/{tool}/)
for sample in SAMPLE_LIST:
    rd = os.path.join(RESULTS_DIR, sample)
    for tool, fname, desc in [
        ("checkv",         "quality_summary.tsv",             "CheckV genome quality"),
        ("phagerbpdetect", f"{sample}_rbp_predictions.tsv",  "Per-protein RBP predictions"),
        ("phagerbpdetect", f"{sample}_rbp_summary.tsv",       "RBP count summary"),
        ("phagetailfinder",f"{sample}_tailfinder.tsv",        "Tail protein classifications"),
        ("phagedpo",       f"{sample}_cds_output.tsv",        "Depolymerase predictions (TSV)"),
        ("phagedpo",       f"{sample}_cds_output.html",       "Depolymerase predictions (HTML)"),
        ("bacphlip",       f"{sample}.fasta.bacphlip",        "BACPHLIP lifestyle prediction"),
        ("rgi",            f"{sample}_rgi.txt",               "RGI AMR hits"),
        ("abricate",       f"{sample}_card.tsv",              "Abricate-CARD AMR screen"),
        ("abricate",       f"{sample}_vfdb.tsv",              "Abricate-VFDB virulence screen"),
        ("acrdb",          f"{sample}_acrdb_blastp.tsv",      "AcrDB BLASTP hits"),
    ]:
        _add(sample, fname, f"results/{sample}/{tool}/", desc,
             os.path.join(rd, tool, fname))

mdf = pd.DataFrame(manifest)
display(HTML(f"<h3>📁 Output Manifest — Run: {RESOLVED_RUN_ID}</h3>"))
styled = (
    mdf.style
    .applymap(lambda v: "color:green;font-weight:bold" if v=="✅" else "color:red;font-weight:bold",
               subset=["status"])
    .format({"size_KB": lambda v: f"{v:,} KB" if pd.notna(v) else "—"})
    .set_properties(**{"font-size":"12px"})
)
display(styled)

n_ok = (mdf["status"] == "✅").sum()
print(f"\n  {n_ok}/{len(mdf)} output files present")
print(f"  All files persist at: /content/drive/MyDrive/{DRIVE_ROOT}/")


---
## Section 9 — Utilities

Convenience cells for uploading FASTAs, downloading results, re-running rules, and debugging environments.


In [ ]:
# ==============================================================================
# CELL 25 — Upload FASTA Files
# Files are saved to Google Drive raw_input/ and persist across sessions.
# ==============================================================================
from google.colab import files
import os

RAW_INPUT_DIR = f"/content/drive/MyDrive/{DRIVE_ROOT}/raw_input"
os.makedirs(RAW_INPUT_DIR, exist_ok=True)

print(f"Upload FASTA files — saved to: {RAW_INPUT_DIR}")
print("Filename must be {sample}.fasta (matching your SAMPLES config).")
print()

uploaded = files.upload()
for fname, data in uploaded.items():
    dst = os.path.join(RAW_INPUT_DIR, fname)
    with open(dst, "wb") as fh:
        fh.write(data)
    print(f"   ✅ {fname} ({len(data)//1024} KB) → {dst}")

print("\nCurrent raw_input/ contents:")
for f in sorted(os.listdir(RAW_INPUT_DIR)):
    sz = os.path.getsize(os.path.join(RAW_INPUT_DIR, f))//1024
    print(f"   {f}  ({sz} KB)")


In [ ]:
# ==============================================================================
# CELL 26 — Download Reports as ZIP
# Packages reports/{run_id}/ as a ZIP and downloads to your local machine.
# ==============================================================================
import subprocess, os
from google.colab import files

REPO_DIR        = "/content/PhagePipeline"
REPORTS_RUN_DIR = os.path.join(REPO_DIR, "reports", RESOLVED_RUN_ID)

if not os.path.exists(REPORTS_RUN_DIR):
    print(f"⚠️  Reports not found at: {REPORTS_RUN_DIR}")
    print("Run the pipeline first (Cell 14).")
else:
    zip_name = f"PhagePipeline_{RESOLVED_RUN_ID}_reports.zip"
    zip_path = f"/tmp/{zip_name}"
    r = subprocess.run(f"zip -r '{zip_path}' .",
                       shell=True, cwd=REPORTS_RUN_DIR,
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f"✅ ZIP created: {zip_path} ({os.path.getsize(zip_path)//1024} KB)")
        files.download(zip_path)
    else:
        print(f"❌ ZIP failed:\n{r.stderr}")


In [ ]:
# ==============================================================================
# CELL 27 — Re-run a Specific Snakemake Rule
# Use this to re-run a single rule after fixing a failure.
# Example FORCE_RULE values: "pharokka", "phold", "score_and_rank"
# ==============================================================================

FORCE_RULE = ""   # <── SET THIS to a rule name from the Snakefile

import subprocess

REPO_DIR = "/content/PhagePipeline"

if not FORCE_RULE.strip():
    print("FORCE_RULE is empty — nothing to do.")
    print("Set FORCE_RULE to a Snakemake rule name (e.g. 'pharokka', 'phold', 'score_and_rank').")
else:
    cmd = SNAKEMAKE_CMD + f" --forcerun {FORCE_RULE.strip()}"
    print(f"Re-running rule: {FORCE_RULE}")
    proc = subprocess.Popen(
        cmd, shell=True, text=True, cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, universal_newlines=True
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode == 0:
        print(f"\n✅ Rule '{FORCE_RULE}' completed.")
    else:
        print(f"\n❌ Rule '{FORCE_RULE}' failed (exit {proc.returncode}).")


In [ ]:
# ==============================================================================
# CELL 28 — Conda Environment Diagnostic
# Verifies each pipeline environment has the correct tools installed.
# ==============================================================================
import subprocess

CONDA_BIN = "/opt/conda/bin/conda"

# (env_name, tool_cmd_to_test)
ENV_CHECKS = [
    ("env_checkv",   "checkv --version"),
    ("env_pharokka", "pharokka.py --version"),
    ("pholdENV",     "phold --version"),
    ("phyntenyENV",  "phynteny_transformer --help 2>&1"),
    ("env_phage_ml", "python -c 'import torch; print(\"torch\", torch.__version__)'"),
    ("env_phage_ml", "python -c 'import transformers; print(\"transformers\", transformers.__version__)'"),
    ("env_phage_ml", "python -c 'import sklearn; print(\"scikit-learn\", sklearn.__version__)'"),
    ("phage",        "python -c 'from Bio import SeqIO; print(\"biopython OK\")'"),
    ("phage",        "python -c 'from pygenomeviz import GenomeViz; print(\"pygenomeviz OK\")'"),
    ("env_bacphlip", "bacphlip --version"),
    ("env_bacphlip", "hmmsearch -h 2>&1"),
    ("env_rgi",      "rgi --version"),
    ("env_abricate", "abricate --version"),
    ("env_acrdb",    "blastp -version"),
    ("env_acrdb",    "makeblastdb -version"),
]

print("Conda Environment Diagnostic")
print("=" * 62)
for env_name, cmd in ENV_CHECKS:
    full = f"{CONDA_BIN} run -n {env_name} {cmd} 2>&1"
    r    = subprocess.run(full, shell=True, capture_output=True, text=True, timeout=30)
    out  = (r.stdout + r.stderr).strip().splitlines()
    icon = "✅" if r.returncode == 0 else "❌"
    short_cmd = cmd.split()[0]
    first_line = out[0][:80] if out else "(no output)"
    print(f"  {icon} [{env_name:12s}] {short_cmd:20s}  →  {first_line}")


---
## Section 10 — Reference

Complete reference for the Colab deployment: directory structure, database paths, conda environments, scoring system, and troubleshooting.


In [ ]:
# ==============================================================================
# CELL 29 — Reference Summary
# ==============================================================================
from IPython.display import display, Markdown

ref = """
## Repository Structure (Colab)

```
/content/PhagePipeline/              ← Cloned from GitHub each session
  Snakefile                          ← Master workflow  (never edit)
  config.yaml                        ← Paths, thresholds, scoring weights
  configs/
    checkv.yaml                      ← env_checkv conda env
    pharokka.yml                     ← env_pharokka conda env
    env_phage_ml.yml                 ← env_phage_ml conda env
  env/
    phold_env.yml                    ← pholdENV conda env
    phynteny_env.yml                 ← phyntenyENV conda env
  scripts/
    checkv/run_checkv.sh             ← DB = $ROOT/databases/checkv-db
    pharokka/run_pharokka.sh         ← DB = $ROOT/databases/pharokka_db
    phage/run_phold.sh               ← DB_PATH = $ROOT/databases/phold_db
    phage/run_phynteny.sh            ← MODELS_PATH = $ROOT/databases/phynteny_models/models
    bacphlip/run_bacphlip.sh         ← uses hmmsearch from env_bacphlip
    rgi/run_rgi.sh                   ← uses --local CARD db in env_rgi
    abricate/run_abricate.sh         ← uses bundled CARD/VFDB in env_abricate
    acrdb/run_acrdb_blast.sh         ← DB = $ROOT/databases/acrdb_db/122_KnownAcr/Known_Acr.faa
    phagedpo/run_phagedpo.sh         ← TOOL = $ROOT/tools/phagedpo/
    phagerbpdetect/run_phagerbpdetect_v4.sh  ← TOOL = $ROOT/tools/PhageRBPdetection/
    phagetailfinder/run_phagetailfinder.sh   ← TOOL = $ROOT/tools/PhageTailFinder/
    build_annotated_gb.py
    build_protein_table.py
    visualize_genome.py
    summarize_sample.py
    score_and_rank.py
  tools/
    PhageRBPdetection/               ← vendored; requires data/RBPdetect_v4_ESMfine/
    PhageTailFinder/                 ← vendored; requires dbs/tail_pfam, dbs/nontail_pfam
    phagedpo/                        ← vendored; phagedpo_cli.py
  databases/  → symlink →  Google Drive/{DRIVE_ROOT}/databases/
  raw_input/  → symlink →  Google Drive/{DRIVE_ROOT}/raw_input/
  results/    → symlink →  Google Drive/{DRIVE_ROOT}/results/
  reports/    → symlink →  Google Drive/{DRIVE_ROOT}/reports/
```

## Google Drive Layout

```
/content/drive/MyDrive/{DRIVE_ROOT}/
  databases/
    checkv-db/                       ← Required: run_checkv.sh  (checkv database-update)
    pharokka_db/                     ← Required: run_pharokka.sh  (install_databases.py)
    phold_db/                        ← Required: run_phold.sh  (phold install-database)
    phynteny_models/models/          ← Required: run_phynteny.sh  (manual download)
    acrdb_db/122_KnownAcr/
      Known_Acr.faa                  ← Required: run_acrdb_blast.sh + config.yaml:acrdb_db
      Known_Acr.faa.p*               ← BLAST index (created by makeblastdb in Cell 9)
  raw_input/
    {sample}.fasta                   ← Place FASTA files here before running
  results/                           ← Raw tool outputs (auto-created)
    {sample}/
      checkv/ pharokka/ phold/ phynteny/ phagedpo/ phagerbpdetect/
      phagetailfinder/ bacphlip/ rgi/ abricate/ acrdb/ logs/
  reports/                           ← Human-facing outputs (auto-created)
    {run_id}/
      phage_scores.tsv
      {sample}/
        {sample}_annotated.gbk
        {sample}_protein_table.tsv
        {sample}_genome_map.png / .svg
        {sample}_tool_summary.tsv
        {sample}_report.txt
```

## Scoring System

| Dimension | Max | Hard disqualifier |
|-----------|-----|-------------------|
| Quality | 20 | No |
| Lifestyle | 30 | Yes — TEMPERATE consensus → REJECTED |
| Safety | 30 | Yes — AMR or Virulence confirmed → REJECTED |
| Lysis | 10 | No |
| Host | 10 | No |

**Thresholds:** PASS ≥ 80 | REVIEW 50–79 | FAIL < 50 | REJECTED

## Troubleshooting

| Symptom | Solution |
|---------|----------|
| PHOLD / PhageRBPdetect very slow | Enable T4 GPU: Runtime → Change runtime type |
| PHOLD fails (no model files) | Verify `databases/phold_db/` on Google Drive |
| PhageRBPdetect fails | Verify `tools/PhageRBPdetection/data/RBPdetect_v4_ESMfine/` |
| Phynteny fails | Verify `databases/phynteny_models/models/` |
| AcrDB BLAST fails: DB not found | Verify Known_Acr.faa + run makeblastdb (Cell 9) |
| Conda env creation fails | Set `FORCE_RECREATE_ENVS = True` and re-run Cell 8 |
| Snakemake reruns completed rules | Symlinks to Drive should preserve timestamps |
| Pipeline hangs on PhageRBPdetect | flock serialises concurrent samples — wait |
| BACPHLIP: hmmsearch not found | Verify hmmer is in env_bacphlip |
| RGI: no hits, card load error | Re-run Cell 9 to reload local CARD database |
"""

display(Markdown(ref))
